# 함수 호출
- 대규모 언어 모델을 외부 도구에 연결하는 방법  

- API 호출에서 함수를 설명하고 모델이 하나 이상의 함수를 호출하기 위한 인수가 포함된 JSON 개체를 출력하도록 지능적으로 선택하도록 할 수 있습니다. Chat Completions API는 함수를 호출하지 않습니다. 대신 모델은 코드에서 함수를 호출하는 데 사용할 수 있는 JSON을 생성합니다.
- 모델은 함수를 호출해야 하는 시기를 감지하고 함수를 준수하는 JSON으로 응답하도록 훈련되었습니다. 이 기능에는 잠재적인 위험도 따릅니다. 사용자를 대신하여 세상에 영향을 미치는 조치(이메일 전송, 온라인 게시, 구매 등)를 수행하기 전에 사용자 확인 흐름을 구축하는 것이 좋습니다.

### 채팅 모델로 함수를 호출하는 방법

함수 호출을 위한 기본 단계는 다음과 같습니다:

1. **모델 호출**: 사용자 질의와 함께 함수 집합을 정의하여 모델을 호출합니다. 함수 목록을 `tools` 파라미터에 제공합니다.


2. **모델의 함수 호출 선택**: 모델은 하나 이상의 함수를 호출할 수 있습니다. 이 경우, 모델의 응답 내용은 사용자가 정의한 스키마에 따른 JSON 객체 문자열이 됩니다. 


3. **JSON 파싱 및 함수 호출**: 코드에서 문자열을 JSON으로 파싱하고, 제공된 인수가 있는 경우 함수를 호출합니다.


4. **모델 재호출 및 결과 요약**: 함수 응답을 새로운 메시지로 추가하여 모델을 다시 호출합니다. 모델이 결과를 사용자에게 요약하여 전달합니다.

이 단계들은 사용자 질의에 따라 적절한 함수를 선택하고, 해당 함수의 응답을 처리하여 최종 결과를 사용자에게 제공하는 과정을 포함합니다. 이를 통해 사용자는 더 나은 응답을 받을 수 있습니다.

<img src="https://i.imgur.com/fR4FuHa.png" width=400 />

### 함수 정의 (Defining Functions)
- 각 API 요청의 tools 매개변수 내에서 함수를 정의할 수 있습니다.
- 함수는 스키마(Schema) 를 통해 정의되며, 모델이 함수의 목적과 입력 인수를 이해할 수 있도록 돕습니다.
스키마는 다음과 같은 필드로 구성됩니다.
| 필드        | 설명                                      |
|------------|------------------------------------------|
| **name**   | 함수의 이름 (예: `get_weather`)          |
| **description** | 함수가 언제, 어떻게 사용되는지 설명  |
| **parameters**  | 함수의 입력 인수를 정의하는 JSON 스키마 |


In [1]:
from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-5-nano"

In [3]:
# 예제 더미 함수로 하드 코딩된 동일한 날씨 정보를 반환합니다.
# 실제 환경에서는 백엔드 API 또는 외부 API가 될 수 있습니다.
def get_current_weather(location, unit="celsius"):
    """지정된 위치의 현재 날씨를 가져옵니다."""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "22.2", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

get_current_weather("Paris")

'{"location": "Paris", "temperature": "22", "unit": "celsius"}'

In [4]:
#함수 집합 정의
tools = [
    {
        "type": "function",   # 도구의 유형을 함수로 설정
        "name": "get_current_weather",   # 함수 이름
        "description": "지정된 위치의 현재 날씨를 가져오는 함수.",    # 함수 설명
        "parameters": {
            "type": "object",    # 파라미터의 유형을 객체로 설정
            "properties": {
                "location": {
                    "type": "string",   # location 파라미터의 데이터 유형을 문자열로 설정
                    "description": "날씨를 알고 싶은 도시와 주 이름. 예: San Francisco, CA",  # location 파라미터 설명
                },
            },
            "required": ["location"],   # location 파라미터는 필수
        },
    }
]

In [5]:
# Step 1: 모델 호출: 사용자 질의와 함께 함수 집합을 정의하여 모델을 호출합니다. 함수 목록을 tools 파라미터에 제공합니다.
# Step 2: 모델은 함수를 호출하기로 결정합니다. 모델은 이름 과 입력 인수를 반환합니다 . 
# 이 경우, 모델의 응답 내용은 사용자가 정의한 스키마에 따른 JSON 객체 문자열이 됩니다.
response = client.responses.create(
    model=Model,
    input=[{"role": "user", "content": "샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?"}],
    tools=tools
)

response.output

[ResponseReasoningItem(id='rs_0c1a5a1c91d21122006a2a075b1278819a856ab4eab0ff8f5d', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"location":"San Francisco, CA"}', call_id='call_WEi5wY2aS2Zb7k5ymog8ssR2', name='get_current_weather', type='function_call', id='fc_0c1a5a1c91d21122006a2a075e75d0819aa3892f543cdfb79f', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Tokyo, Japan"}', call_id='call_vBk72gTkhuAd2c8n0TmooUxO', name='get_current_weather', type='function_call', id='fc_0c1a5a1c91d21122006a2a075e75e0819a81c1b65719f43615', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Paris, France"}', call_id='call_jJXZbe5nZBbZBD6uuWM6oH4r', name='get_current_weather', type='function_call', id='fc_0c1a5a1c91d21122006a2a075e75e8819a9b1b7d52b6e17700', status='completed')]

In [6]:
input_messages = [{"role": "user", "content": "샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?"}]

for item in response.output:
    if item.type == "reasoning":
        # reasoning 아이템도 함께 추가
        input_messages.append(item)
    elif item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_current_weather(args['location'])
        input_messages.append(item)
        input_messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": str(result)
        })

input_messages

[{'role': 'user', 'content': '샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?'},
 ResponseReasoningItem(id='rs_0c1a5a1c91d21122006a2a075b1278819a856ab4eab0ff8f5d', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"location":"San Francisco, CA"}', call_id='call_WEi5wY2aS2Zb7k5ymog8ssR2', name='get_current_weather', type='function_call', id='fc_0c1a5a1c91d21122006a2a075e75d0819aa3892f543cdfb79f', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_WEi5wY2aS2Zb7k5ymog8ssR2',
  'output': '{"location": "San Francisco", "temperature": "22.2", "unit": "celsius"}'},
 ResponseFunctionToolCall(arguments='{"location":"Tokyo, Japan"}', call_id='call_vBk72gTkhuAd2c8n0TmooUxO', name='get_current_weather', type='function_call', id='fc_0c1a5a1c91d21122006a2a075e75e0819a81c1b65719f43615', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_vBk72gTkhuAd2c8n0TmooUxO',
  'output': '{"location": "Tokyo", "temperatur

In [7]:
# step 4: 결과가 포함된 모델을 제공하여 최종 대응에 반영할 수 있습니다.
response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)
print(response_2.output_text)

다음은 현재 날씨 정보입니다.

- 샌프란시스코: 약 22°C
- 도쿄: 약 10°C
- 파리: 약 22°C

원하시면 하늘 상태, 바람, 습도, 강수 확률 등 추가 정보나 화씨로도 변환해 드릴게요.


### Free Weather API 이용
https://open-meteo.com/

In [8]:
import requests

# 현재의 온도 가져오기
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = response.json()
    return data['current']['temperature_2m']

# 서울의 위도, 경도
get_weather(37.56667, 126.97806)

19.7

In [9]:
import json

tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "제공된 좌표의 현재 기온을 섭씨(Celsius) 단위로 가져오세요.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True   # JSON 스키마의 엄격한 준수(strict mode)를 적용
    }]

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[{"role": "user", "content": "오늘 서울의 기온이 어때?"}],
    tools=tools,
)

response.output

[ResponseFunctionToolCall(arguments='{"latitude":37.5665,"longitude":126.978}', call_id='call_0cscAw2updP7DD1yfNu4LgkM', name='get_weather', type='function_call', id='fc_0ab234460c618697006a2a0766d470819aa50ea9c2e918e867', status='completed')]

In [10]:
# 함수 호출 실행
tool_call = response.output[0]
args = json.loads(tool_call.arguments)

result = get_weather(args['latitude'], args['longitude'])
result

19.7

In [11]:
# 함수 호출 실행 결과를 추가합니다.
input_messages = [{"role": "user", "content": "오늘 서울의 기온이 어때?"}]
input_messages.append(tool_call)
input_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)

response_2.output_text

'현재 서울의 기온은 약 19.7°C입니다.\n\n오늘은 비교적 선선한 편이니 가벼운 겉옷 정도를 준비하시면 좋습니다. 원하시면 습도나 오늘의 날씨 예보도 함께 알려드릴게요.'

### Email 보내기

In [17]:
# email 생성 및 실제 발송 함수 (Gmail SMTP 이용)
# 이 함수 하나를 Responses API 예제와 Agents SDK 예제에서 공통으로 사용한다.
import os
import smtplib, ssl
from email.utils import formatdate
from email.mime.multipart import MIMEMultipart
from email.header import Header
from email.mime.text import MIMEText

# 보내는 사람 계정 정보
SENDER_EMAIL = "ironmanciti@gmail.com"   # 보내는 사람 Gmail 주소 (본인 계정으로 변경)
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465  # SSL 포트

def send_email(to: str, subject: str, body: str) -> str:
    """지정된 수신자에게 제목과 내용을 포함한 이메일을 Gmail SMTP 로 실제 발송한다.

    Args:
        to: 수신자 이메일 주소
        subject: 이메일 제목
        body: 이메일 본문 내용
    """
    # .env 에 저장된 Gmail 앱 비밀번호(smtp_password)를 읽어옴
    password = os.environ.get("smtp_password")
    if not password:
        raise ValueError("환경변수 smtp_password 가 설정되어 있지 않습니다. .env 파일을 확인하세요.")

    msg = MIMEMultipart()
    msg['From'] = SENDER_EMAIL
    msg['To'] = to
    msg['Date'] = formatdate(localtime=True)
    # Header 클래스를 사용해야 한글 제목 인코딩 문제가 발생하지 않음
    msg['Subject'] = Header(s=subject, charset='utf-8')
    msg.attach(MIMEText(body, _charset='utf-8'))

    # SMTP_SSL 로 Gmail 서버에 접속하여 실제 발송
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, context=context) as server:
        server.login(SENDER_EMAIL, password)
        server.sendmail(SENDER_EMAIL, to, msg.as_string())

    return f"{to} 에게 성공적으로 메일 발송"

In [13]:
# 날씨 조회(get_weather) + 이메일 발송(send_email) 두 도구를 함께 제공
# 모델이 부산 날씨를 먼저 조회한 뒤, 그 결과를 바탕으로 이메일 본문을 작성하여 발송하도록 함
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "제공된 좌표(위도, 경도)의 현재 기온을 섭씨(Celsius) 단위로 가져오세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number", "description": "위도"},
                "longitude": {"type": "number", "description": "경도"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "지정된 수신자에게 제목과 메시지를 포함한 이메일을 보내세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "수신자의 이메일 주소"
                },
                "subject": {
                    "type": "string",
                    "description": "이메일 제목 줄"
                },
                "body": {
                    "type": "string",
                    "description": "이메일 본문"
                }
            },
            "required": [
                "to",
                "subject",
                "body"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
]

messages = [{"role": "user",
    "content": (
        "먼저 현재 부산의 날씨를 get_weather 함수로 알아낸 뒤, "
        "그 날씨에 어울리는 부산 여행 방문지와 복장에 대해 조언하는 이메일을 작성해서 "
        "trimurti@naver.com 과 youngjea.oh@oyj.ai.kr 에게 각각 보내줄 수 있나요?"
    )}]

response = client.responses.create(
    model=Model,
    input=messages,
    tools=tools
)

response.output

[ResponseReasoningItem(id='rs_061176e5130b2296006a2a076f83a8819a889d116e61b956a2', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"latitude":35.18,"longitude":129.07}', call_id='call_I08fj7XK2C659EkcnsTbSkud', name='get_weather', type='function_call', id='fc_061176e5130b2296006a2a0773a060819a8bc7eab6de697f09', status='completed')]

In [14]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "send_email":
        return send_email(**args)

In [15]:
# 함수 호출이 더 이상 없을 때까지 반복 실행
# (1라운드: get_weather 로 부산 날씨 조회 -> 2라운드: send_email 로 이메일 작성/발송 -> 3라운드: 최종 자연어 응답)
input_messages = messages.copy()  # 원본 messages 리스트 수정 방지를 위해 복사

while True:
    # 모델 출력(reasoning + function_call)을 모두 대화 기록에 추가
    # reasoning 아이템과 function_call 아이템을 쌍으로 유지해야 하므로 전체를 추가
    input_messages += response.output

    # 이번 응답에 포함된 function_call 들만 추출
    tool_calls = [item for item in response.output if item.type == "function_call"]
    if not tool_calls:
        break  # 더 이상 호출할 함수가 없으면 (모델이 최종 답변을 생성했으면) 종료

    # 각 function_call 을 실제로 실행하고 결과를 대화 기록에 추가
    for tool_call in tool_calls:
        name = tool_call.name                    # 모델이 호출하려는 함수 이름
        args = json.loads(tool_call.arguments)   # JSON 문자열 인자를 dict 로 파싱
        result = call_function(name, args)       # 실제 함수 실행 (get_weather / send_email)
        input_messages.append({
            "type": "function_call_output",      # 함수 실행 결과 메시지
            "call_id": tool_call.call_id,        # 어느 function_call 에 대한 결과인지 연결
            "output": str(result)
        })

    # 함수 실행 결과를 포함해 모델을 다시 호출 (다음 라운드)
    response = client.responses.create(
        model=Model,
        input=input_messages,
        tools=tools,
    )

print(response.output_text)  # 모델의 최종 자연어 응답 출력

네, 처리했습니다.

요약
- 현재 부산 기온: 약 23.1°C
- 두 명의 수신처로 동일한 내용의 이메일 발송
  - 수신자: trimurti@naver.com
  - 수신자: youngjea.oh@oyj.ai.kr
- 이메일 제목: 부산 현재 날씨에 맞춘 당일 여행지 추천 및 복장 팁
- 이메일 내용(핵심 요약):
  - 부산 현재 기온 안내 및 야외 활동에 적합한 날씨라는 점 안내
  - 방문지 추천: 해운대 해수욕장 + 동백섬, 광안리 해변, 감천문화마을, 자갈치시장/국제시장
  - 복장 팁: 반팔 티셔츠/반바지 또는 얇은 긴팔 상의, 편한 운동화, 선글라스와 모자, 자외선 차단제, 저녁이나 바람 있을 때를 대비한 가벼운 자켓

다른 요청이 있으시면 알려주세요
- 수신자별로 서로 다른 내용으로 맞춤 메일 작성
- 일정이나 예산에 맞춘 코스 조정
- 특정 날짜에 맞춘 일정 제안이나 추가 방문지 추천


## Agent SDK 를 이용하여 동일한 기능 구현

In [16]:
from agents import Agent, Runner, function_tool

# 위에서 정의한 send_email 함수를 그대로 Agents SDK 도구로 등록한다.
# function_tool(send_email) 은 함수의 타입 힌트와 docstring 을 읽어 도구 스키마를 자동 생성하므로
# 동일한 SMTP 발송 로직을 다시 작성할 필요가 없다. (중복 제거)
agent = Agent(
    name="Email Assistant",
    instructions=(
        "당신은 유용한 이메일 도우미입니다. "
        "사용자가 이메일 전송을 요청하면 send_email 도구를 사용하세요. "
        "여러 명에게 보내야 할 경우 각각 따로 함수를 호출하세요."
    ),
    model=Model,
    tools=[function_tool(send_email)]
)

result = await Runner.run(
    agent, 
    input=(
        "먼저 현재 부산의 날씨를 get_weather 함수로 알아낸 뒤, "
        "그 날씨에 어울리는 부산 여행 방문지와 복장에 대해 조언하는 이메일을 작성해서 "
        "trimurti@naver.com 과 youngjea.oh@oyj.ai.kr 에게 각각 보내줄 수 있나요?"
    )
)
print(result.final_output)

현재 부산의 날씨를 실제로 확인하는 get_weather 함수나 API에 접근할 수 있는 상태가 아니라서, 정확한 정보를 바탕으로 이메일을 보내드리려면 먼저 날씨 데이터를 알려주셔야 합니다. 원하신다면 placeholder(예시)로 작성해 두고, 실제 날씨가 확인되는 즉시 업데이트해 보내드릴 수도 있습니다.

다음 중 어떤 방식으로 진행할지 알려 주세요:
- 옵션 A: 현재 날씨 정보를 직접 보내주시면 그에 맞춰 두 이메일을 맞춤형으로 작성하고 즉시 발송해 드립니다.
- 옵션 B: 예시로 사용할 임시 날씨를 가정해 이메일 초안을 작성해 두고, 실제 날씨로 업데이트해 발송해 드리길 원하시면 알려 주세요. 예시로는 예를 들어 “현재 부산의 날씨: 맑음, 약 25°C, 바람 약간” 같은 형식을 사용합니다.

참고로 아래는 두 수신인에 보낼 이메일 초안 예시입니다. 현재 날씨 정보를 반영해 텍스트를 교정해 드릴 수 있습니다.

- 수신자 1: trimurti@naver.com
  제목: 부산 날씨에 맞춘 당일 방문지와 복장 조언
  본문 예시:
  안녕하세요, Trimurti 님.
  오늘의 부산 날씨를 바탕으로 당일 방문지와 복장을 정리해 드립니다.
  현재 날씨: [예: 맑음, 약 XX°C, 바람 약간]
  추천 방문지(날씨 걱정 없이 편하게 다닐 수 있는 코스)
  - 자갈치 시장 및 남포동 산책
  - 부산 타워/해운대 근처 산책로
  - 감천문화마을
  - BIFF 광장 근처 카페
  복장 및 준비물
  - 햇빛 차단용 모자, 선크림
  - 가벼운 겉옷(온도에 따라), 편한 신발
  - 물병
  참고: 실제 현재 날씨로 맞춤화해 드리려면 오늘의 날씨를 보내주시면 즉시 업데이트하겠습니다.
  감사합니다.

- 수신자 2: youngjea.oh@oyj.ai.kr
  제목: 부산 당일 방문지와 복장 조언(오늘의 날씨 반영)
  본문 예시:
  안녕하세요, Youngjea 님.
  오늘 부산 날씨를 반영한 당일 방문지와 복장 팁을 정리했습니다.
  현재 날씨: [예